# 04 — XGBoost の核心：2階近似、葉の値、分割利得
## 1. 逐次加法モデル
時点 $t$ で、新しい木 $f_t$ を追加する:
$$\hat y_i^{(t)}=\hat y_i^{(t-1)}+f_t(x_i).$$

近似する目標（定数項を除く）は

$$\tilde{\mathcal L}^{(t)}
=\sum_i\left[g_if_t(x_i)+\frac12h_if_t(x_i)^2\right]
+\gamma T+\frac{\lambda}{2}\sum_{j=1}^T w_j^2,$$

$$g_i=\partial_{\hat y}\ell(y_i,\hat y_i^{(t-1)}),\qquad
h_i=\partial^2_{\hat y}\ell(y_i,\hat y_i^{(t-1)}).$$

$T$ は葉数、$w_j$ は葉の出力、$\lambda\ge0$ は L2 正則化、$\gamma\ge0$ は葉数ペナルティ。

**注意:** これは損失の Taylor 2 次近似。決定木の構造は離散なので、勾配降下で直接 split 構造を微分するわけではない。

## 2. 葉ごとに集約して最小化
葉 $j$ に入るサンプル集合を $I_j$ とし

$$G_j=\sum_{i\in I_j}g_i,\quad H_j=\sum_{i\in I_j}h_i.$$

葉の目的関数は

$$G_jw_j+\frac12(H_j+\lambda)w_j^2.$$

微分して0と置けば

$$\boxed{w_j^*=-\frac{G_j}{H_j+\lambda}}.$$

代入すると木の最適目的関数（定数項を除く）は
$$\tilde{\mathcal L}^*=-\frac12\sum_j\frac{G_j^2}{H_j+\lambda}+\gamma T.$$

$H_j+\lambda>0$ と仮定。これを使って、「ここで split するべきか」を定量化する。

## 3. Split gain
ある葉を左右に割るとき
$$\boxed{
\mathrm{Gain}=\frac12\left[
\frac{G_L^2}{H_L+\lambda}+
\frac{G_R^2}{H_R+\lambda}-
\frac{(G_L+G_R)^2}{H_L+H_R+\lambda}
\right]-\gamma.
}$$

Gain が正なら近似目的が改善する。$\gamma$ が大きいほど小さな改善による分割を抑える。$\lambda$ が大きいほど葉の重みの絶対値が縮む。

**手計算:** $G_L=-4$, $G_R=4$, $H_L=H_R=2$, $\lambda=2$ のとき Gain = $4-\gamma$。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
G,H=4,2
lambdas=[0,1,5]
ws=np.linspace(-3,1,201)
for lam in lambdas:
    objective=G*ws+0.5*(H+lam)*ws**2
    plt.plot(ws,objective,label=f"lambda={lam}; optimum={-G/(H+lam):.2f}")
plt.xlabel("leaf weight w");plt.ylabel("local quadratic objective")
plt.title("L2 regularization shrinks leaf weights")
plt.legend();plt.show()

## 4. 分類の勾配と Hessian
二値分類で margin（logit） $z$ と $p=\sigma(z)=1/(1+e^{-z})$ を考える。損失を1サンプルの binary cross-entropy とすると

$$\ell(y,z)=-y\log p-(1-y)\log(1-p),\qquad
g=p-y,\quad h=p(1-p).$$

ここで \`h\` は 2階の曲率であって 2つ目のモデルや学習率ではない。極端な logit で overflow しない sigmoid を実装するとよい。

**学習すべき差:** Gradient Boosting の平方誤差では残差が負の勾配、XGBoost の木は $g,h$ を葉ごとに集約し、正則化された2次目的を減らす。

実際の XGBoost には histogram 近似や欠損方向の学習などの実装技術もあるが、本章では数学的な中心部分に限定する。

In [ ]:
logits = np.array([-2.,0.,2.])
p = 1/(1+np.exp(-logits))
y = np.array([0.,1.,1.])
print("p:", p)
print("g:", p-y)
print("h:", p*(1-p))
print('採点: python tools/grade.py -k "test_optimal_leaf_weight or test_split_gain or test_logistic"')